## Statistics of Lyman-α profiles, metal emission lines and absorption lines

This notebook generates histograms and scatter plots of key parameters in the megatable, and searches for relationships between them

In [ ]:
from astropy.io import fits
import astropy.table as aptb

# Load the megatab
SPEC_SOURCE = "R21"  #  Where are the spectra from? 'R21' for Richard+21, 'APER' for apertures
SPEC_TYPE   = "weight_skysub"

tabfile = f"megatables/lae_megatab_flagged_cpts_allrefit_zeldamcmc_sysz_absv_{SPEC_TYPE}.fits"
tabhdul = fits.open(tabfile)
megatable = aptb.Table(tabhdul[1].data)

# Only keep sources with significant Lyman alpha detection
megatab = megatable[(megatable['SNRR'] > 5.0) | (megatable['SNRB'] > 5.0)]

In [ ]:
from tangelo.statistics import make_scatter
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import linregress
from tangelo.spectroscopy import muse_lsf_fwhm_poly
from tangelo.constants import wavedict


# Helper functions to get line any lyman alpha properties that aren't directly in the megatable, and to apply rest-frame correction if needed.
def get_line_property(megatab: aptb.Table, line: str, prop: str, 
                      rest_frame: bool = True, correct_inst: bool = True,
                      abs: bool = False) -> tuple[np.ndarray, np.ndarray]:
    """
    Helper function to get the line property column from the megatable, handling parameters that aren't
    directly in the megatable (e.g. LYA_EW, BRRATIO, DISPR, FWHMR) and applying rest-frame correction if needed.
    Note that absorption lines have their equivalent widths flipped to maintain the convention of positive values 
    indicating stronger absorption, which is important for interpreting correlations with Lyman alpha properties.

    Parameters
    ----------
    megatab : astropy.table.Table
        The megatable containing the data.
    line : str
        The name of the line (e.g. 'CIV1548', 'SiII1260').
    prop : str
        The property to retrieve (e.g. 'EW', 'FWHM').
    rest_frame : bool, optional
        Whether to apply rest-frame correction for wavelength-based properties, by default True.
    correct_inst : bool, optional
        Whether to apply instrumental resolution correction for FWHM using the MUSE LSF polynomial,
        by default True.
    abs : bool, optional
        Whether the line is an absorption line, which requires flipping the sign of the equivalent width
        to be consistent with the convention that positive values indicate stronger absorption

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        The requested line property column and its associated error column.
    """
    col_name = f"{prop}_{line}"
    err_name = f"{prop}_ERR_{line}"
    if prop == "EW":
        # Calculate EW from FLUX, FLUX_ERR, CONT and CONT_ERR if not directly available
        flux_col = f"FLUX_{line}"
        flux_err_col = f"FLUX_ERR_{line}"
        cont_col = f"CONT_{line}"
        cont_err_col = f"CONT_ERR_{line}"
        if flux_col in megatab.colnames and flux_err_col in megatab.colnames and cont_col in megatab.colnames and cont_err_col in megatab.colnames:
            flux = megatab[flux_col].copy()
            flux_err = megatab[flux_err_col].copy()
            cont = megatab[cont_col].copy()
            cont_err = megatab[cont_err_col].copy()
            ew = flux / cont
            ew_err = np.abs(ew * np.sqrt((flux_err / flux)**2 + (cont_err / cont)**2))
            if rest_frame:
                ew /= (1 + megatab['z'])
                ew_err /= (1 + megatab['z'])
            if abs:
                ew *= -1  # Flip sign for absorption lines to maintain convention of positive values 
                            # indicating stronger absorption
            return ew, ew_err
        else:
            raise ValueError(f"Required columns for calculating EW of {line} not found in megatable.")
    elif prop == "FWHM":
        # Calculate FWHM from FWHM_OPT and redshift if not directly available
        fwhm_col = f"FWHM_{line}"
        if fwhm_col in megatab.colnames and err_name in megatab.colnames:
            fwhm = megatab[fwhm_col].copy()
            fwhm_err = megatab[err_name].copy()
            # Apply instrumental resolution correction using MUSE LSF polynomial if not already applied
            if correct_inst:
                fwhm, fwhm_err = correct_inst_res(fwhm, fwhm_err, megatab[f'LPEAK_{line}'], "FWHM")
            if rest_frame:
                fwhm /= (1 + megatab['z'])
                fwhm_err /= (1 + megatab['z'])
            return fwhm, fwhm_err
        else:
            raise ValueError(f"Required column for calculating FWHM of {line} not found in megatable.")
    elif prop == "CVEL":
        # Calculate velocity centroid of the line relative to the systemic redshift, which can
        # be derived from the Lyman alpha peak redshift LPEAKR and the offset from systemic
        # DELTAV_LYA
        lya_z = megatab['LPEAKR'] / 1215.67 - 1
        sys_z = lya_z - megatab['DELTAV_LYA'] / 299792.458 * (1 + lya_z)  # Convert velocity offset to redshift
        line_peak_rest = megatab[f'LPEAK_{line}'] / (1 + sys_z)  # Convert observed line peak to rest-frame wavelength
        line_peak_rest_err = megatab[f'LPEAK_ERR_{line}'].copy() / (1 + sys_z)  # Propagate error through rest-frame conversion
        line_rest_wave = wavedict[line]  # Get rest-frame wavelength of the line from wavedict
        cvel = (line_peak_rest - line_rest_wave) / line_rest_wave * 299792.458  # Convert to velocity offset from rest wavelength
        cvel_err = line_peak_rest_err / line_rest_wave * 299792.458  # Propagate error through velocity conversion
        return cvel, cvel_err
    elif col_name in megatab.colnames and err_name in megatab.colnames:
        return megatab[col_name].copy(), megatab[err_name].copy()
    else:
        raise ValueError(f"Column {col_name} not found in megatable.")
    
def correct_inst_res(col: np.ndarray, col_err: np.ndarray, lpeakr: np.ndarray, prop: str) -> tuple[np.ndarray, np.ndarray]:
    """
    Apply instrumental resolution correction for FWHM and DISP using the MUSE LSF polynomial,
    with proper error propagation.

    Parameters
    ----------
    col : np.ndarray
        The column to correct (FWHM or DISP).
    col_err : np.ndarray
        The error on the column to correct.
    lpeakr : np.ndarray
        The observed wavelength of the red peak of Lyman alpha, used to determine the 
        instrumental resolution from the MUSE LSF polynomial.
    prop : str
        The property being corrected ('FWHM' or 'DISP').

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        The instrumentally corrected column and its propagated error.
    """
    lsf_fwhm = muse_lsf_fwhm_poly(lpeakr)
    
    if prop == "DISP":
        lsf = lsf_fwhm / (2 * np.sqrt(2 * np.log(2)))  # Convert FWHM to dispersion
    elif prop == "FWHM":
        lsf = lsf_fwhm
    else:
        raise ValueError("Invalid property for instrumental correction. Must be 'FWHM' or 'DISP'.")
    
    # Store original values for error propagation
    col_obs = col.copy()
    
    # Quadrature subtraction: corrected = sqrt(obs^2 - lsf^2)
    corrected_col = np.sqrt(np.maximum(col_obs**2 - lsf**2, 0))
    
    # Error propagation: d(corrected)/d(obs) = obs / corrected
    # Handle cases where corrected_col is near zero to avoid division by zero
    with np.errstate(divide='ignore', invalid='ignore'):
        err_factor = col_obs / np.maximum(corrected_col, 1e-30)
        corrected_err = col_err * err_factor
    
    # Set error to large value where correction was forced to zero (obs < lsf)
    invalid_mask = (col_obs**2 - lsf**2) < 0
    corrected_err[invalid_mask] = np.inf
    
    return corrected_col, corrected_err
        
def get_lya_property(megatab: aptb.Table, prop: str, rest_frame: bool = True,
                     correct_inst: bool = True) -> tuple[np.ndarray, np.ndarray]:
    """
    Helper function to get the Lyman alpha property column from the megatable, handling parameters that aren't
    directly in the megatable (e.g. LYA_EW, BRRATIO, DISPR, FWHMR).

    Parameters
    ----------
    megatab : astropy.table.Table
        The megatable containing the data.
    prop : str
        The Lyman alpha property to retrieve (e.g. 'EW', 'BRRATIO', 'DISPR', 'FWHMR').
    rest_frame : bool, optional
        Whether to apply rest-frame correction for wavelength-based properties, by default True.
    correct_inst : bool, optional
        Whether to apply instrumental resolution correction for FWHMR and DISPR using the MUSE
        LSF polynomial, by default True.

    Returns
    -------
    tuple[np.ndarray, np.ndarray]
        The requested Lyman alpha property column and its associated error column.
    """
    if prop in ["LYA_EW", "EW"]:
        # Calculate Lya EW from FLUXR, FLUXR_ERR, CONT and CONT_ERR if not directly available
        fluxr_col = "FLUXR"
        fluxr_err_col = "FLUXR_ERR"
        fluxb_col = "FLUXB"
        fluxb_err_col = "FLUXB_ERR"
        cont_col = "CONT"
        cont_err_col = "CONT_ERR"
        if fluxr_col in megatab.colnames and fluxr_err_col in megatab.colnames and fluxb_col in megatab.colnames and fluxb_err_col in megatab.colnames and cont_col in megatab.colnames and cont_err_col in megatab.colnames:
            fluxr = megatab[fluxr_col].copy()
            fluxr_err = megatab[fluxr_err_col].copy()
            fluxb = megatab[fluxb_col].copy()
            fluxb_err = megatab[fluxb_err_col].copy()
            # replace any NaN fluxb and fluxb_err values with 0 to avoid issues in EW calculation for sources without significant blue peak detection
            fluxb = np.nan_to_num(fluxb, nan=0.0)
            fluxb_err = np.nan_to_num(fluxb_err, nan=0.0)
            flux_total = fluxr + fluxb
            flux_total_err = np.sqrt(fluxr_err**2 + fluxb_err**2)
            cont = megatab[cont_col].copy()
            cont_err = megatab[cont_err_col].copy()
            ew = flux_total / cont
            ew_err = np.abs(ew * np.sqrt((flux_total_err / flux_total)**2 + (cont_err / cont)**2))
            if rest_frame:
                ew /= (1 + megatab['z'])  # Rest-frame correction
                ew_err /= (1 + megatab['z'])
            return ew, ew_err
        else:
            raise ValueError("Required columns for calculating Lya EW not found in megatable.")
    elif prop == "BRRATIO":
        # Calculate blue-to-red flux ratio from FLUXB and FLUXR if not directly available
        blue_flux_col = "FLUXB"
        red_flux_col = "FLUXR"
        if blue_flux_col in megatab.colnames and red_flux_col in megatab.colnames:
            blue_flux = megatab[blue_flux_col].copy()
            red_flux = megatab[red_flux_col].copy()
            with np.errstate(divide='ignore', invalid='ignore'):
                br_ratio = blue_flux / red_flux
                br_ratio[red_flux == 0] = np.nan  # Avoid division by zero
            br_ratio_err = br_ratio * np.sqrt((megatab["FLUXB_ERR"].copy() / blue_flux)**2 
                                              + (megatab["FLUXR_ERR"].copy() / red_flux)**2)
            return br_ratio, br_ratio_err
        else:
            raise ValueError("Required columns for calculating Lya blue-to-red flux ratio not found in megatable.")
    elif prop == "BRSEP":
        # Calculate blue-red peak separation from LPEAKR and LPEAKB if not directly available
        red_peak_col = "LPEAKR"
        blue_peak_col = "LPEAKB"
        if red_peak_col in megatab.colnames and blue_peak_col in megatab.colnames:
            red_peak = megatab[red_peak_col].copy()
            blue_peak = megatab[blue_peak_col].copy()
            if rest_frame:
                br_sep = (red_peak - blue_peak) / (1 + megatab['z'])  # Rest-frame correction
                br_sep_err = np.sqrt(megatab["LPEAKR_ERR"].copy()**2 + megatab["LPEAKB_ERR"].copy()**2) / (1 + megatab['z'])
            else:
                br_sep = red_peak - blue_peak
                br_sep_err = np.sqrt(megatab["LPEAKR_ERR"].copy()**2 + megatab["LPEAKB_ERR"].copy()**2)
            br_sep[(red_peak == 0) | (blue_peak == 0)] = np.nan  # Avoid invalid values
            return br_sep, br_sep_err
        else:
            raise ValueError("Required columns for calculating Lya blue-red peak separation not found in megatable.")
    elif prop[:-1] in ["DISP", "FWHM", "FWHM_AB"] and prop in megatab.colnames:
        err_name = f"{prop}_ERR"
        # For these parameters, just apply rest-frame correction to the existing column
        col = megatab[prop].copy()
        col_err = megatab[err_name].copy()
        if correct_inst:
            lpeakr_col = "LPEAKR"
            if lpeakr_col in megatab.colnames:
                lpeakr = megatab[lpeakr_col].copy()
                lsf_fwhm = muse_lsf_fwhm_poly(lpeakr)
                if prop[:-1] == "DISP":
                    lsf_disp = lsf_fwhm / (2 * np.sqrt(2 * np.log(2)))  # Convert FWHM to dispersion
                    col, col_err = correct_inst_res(col, col_err, lpeakr, "DISP")
                else:  # FWHM or FWHM_AB
                    col, col_err = correct_inst_res(col, col_err, lpeakr, "FWHM")
            else:
                raise ValueError("Required column for instrumental resolution correction not found in megatable.")
        if rest_frame:
            col /= (1 + megatab['z'])  # Rest-frame correction
            col_err /= (1 + megatab['z'])
        return col, col_err
    elif "ZELDA" in prop and prop in megatab.colnames:
        # for ZELDA parameters, no need to apply corrections; however, the error bars are
        # stored in two separate columns (ERRM and ERRP) for negative and positive errors, 
        # so we take the average of these for simplicity in plotting and correlation analysis
        prop_base = prop.rsplit('_', 1)[0]  # Get the base property name without the ZELDA suffix
        errm_name = f"{prop_base}_ERRM_ZELDA"
        errp_name = f"{prop_base}_ERRP_ZELDA"
        if errm_name in megatab.colnames and errp_name in megatab.colnames:
            col_err = (megatab[errm_name].copy() + megatab[errp_name].copy()) / 2
            return megatab[prop].copy(), col_err
        else:
            return megatab[prop].copy(), None
    elif prop in megatab.colnames:
        err_name = f"{prop}_ERR"
        if err_name in megatab.colnames:
            return megatab[prop].copy(), megatab[err_name].copy()
        else:
            return megatab[prop].copy(), None
    else:
        raise ValueError(f"Column {prop} not found in megatable.")
    
def prepare_scatter_mask(megatab: aptb.Table, line: str, line_col: np.ndarray, line_prop: str,
                         lya_col: np.ndarray, lya_prop: str, abs_lines: list[str],
                         include_upper_limits: bool = False, sig_thresh: float = 3.0) -> np.ndarray:
    """
    Prepare a mask for scatter plot analysis between a given line and Lyman alpha property.

    Parameters
    ----------
    megatab : aptb.Table
        The megatable containing the data.
    line : str
        The line to analyze (e.g., "SiII1260").
    line_col : np.ndarray
        The column for the line property (e.g., equivalent width or FWHM).
    line_prop : str
        The line property being analyzed (e.g., "EW", "FWHM").
    lya_col : np.ndarray
        The Lyman alpha property column.
    lya_prop : str
        The Lyman alpha property to analyze (e.g., "LYA_EW").
    abs_lines : list[str]
        List of lines that should be treated as absorption.
    include_upper_limits : bool, optional
        Whether to include upper limits for for non-detections. If True, sources with non-significant detections
        are not masked.
    sig_thresh : float, optional
        The significance threshold (in sigma) for including sources based on their SNR

    Returns
    -------
    np.ndarray
        A boolean mask indicating valid data points for scatter plot analysis.
    """
    mask = np.isfinite(line_col) & np.isfinite(lya_col)
    # Only include sources with significant line and continuum detection
    if not include_upper_limits:
        # For emission, require SNR > sig_thresh, for absorption, require < -sig_thresh
        mask &= (megatab[f"SNR_{line}"] > sig_thresh) if line not in abs_lines else (megatab[f"SNR_{line}"] < -sig_thresh)
    else:
        # If including upper limits, only require that the line is not significantly detected in the opposite direction
        mask &= (megatab[f"SNR_{line}"] > -sig_thresh) if line not in abs_lines else (megatab[f"SNR_{line}"] < sig_thresh)
    
    if line_prop == "EW":
        # Always require significant continuum for EW measurements as without this you can't even get upper bounds
        mask &= (megatab[f"CONT_{line}"] / megatab[f"CONT_ERR_{line}"] > sig_thresh)
    elif line_prop in ["FWHM", "CVEL"]:
        mask &= line_col > 0  # Only consider positive FWHM and CVEL values to avoid unphysical results from poor fits
    
    # Only include unflagged lines
    mask &= (megatab[f"FLAG_{line}"] == '')
    # If fitting Lya EW or CONT, require significant continuum detection to ensure reliable EW measurement
    if lya_prop in ["LYA_EW", "CONT", "EW"]:
        mask &= (megatab['CONT'] / megatab['CONT_ERR'] > sig_thresh)
    
    # Only take positive Lya ASYMR values to focus on sources with stronger red peaks, which are more likely to have reliable Lya EW measurements and be less affected by IGM absorption.
    if lya_prop in ["ASYMR", "DELTAV_LYA", "FWHMR", "DISPR", "VEXP_ZELDA"]:
        mask &= lya_col > 0
    
    return mask

In [ ]:
from typing import Optional
from linmix import LinMix
from scipy.odr import ODR, Model, RealData
from scipy import stats

def get_mcmc_p_value(chain: np.ndarray) -> float:
    """
    Calculate a p-value from the MCMC posterior distribution of the slope parameter (beta) in LinMix.
    """
    # Calculate probability of positive/negative slope
    prob_positive = np.mean(chain['beta'] > 0)
    prob_negative = 1 - prob_positive

    # For two-tailed test, this is the probability of the opposite sign
    p_value = 2 * min(prob_positive, prob_negative)

    # Add a warning if p_value hits resolution limit
    min_possible_p = 2.0 / len(chain)
    if p_value <= min_possible_p * 1.1:  # Within 10% of minimum
        print(f"Warning: p-value ({p_value:.2e}) is at resolution limit. "
            f"Consider longer MCMC run for more precision. "
            f"All {len(chain)} posterior samples have the same sign.")

    return p_value

def do_linregress(x: np.ndarray, y: np.ndarray, x_err: np.ndarray, y_err: np.ndarray,
                  mcmc: bool = True, ax_in: Optional['matplotlib.axes.Axes'] = None, 
                  delta: Optional[np.ndarray] = None, log_transformed: bool = False) -> tuple[float, float, float, float, float]:
    """
    Perform linear regression using either the LinMix MCMC method, which accounts for measurement errors in both x and y, 
    or a simple ODR regression if MCMC is disabled.

    Parameters
    ----------
    x : np.ndarray
        The x-values of the data points.
    y : np.ndarray
        The y-values of the data points.
    x_err : np.ndarray
        The uncertainties in the x-values.
    y_err : np.ndarray
        The uncertainties in the y-values.
    mcmc : bool, optional
        Whether to perform MCMC regression using LinMix, by default True. If False, will
        perform a simple ODR regression.
    ax_in : matplotlib.axes.Axes, optional
        An optional matplotlib Axes object to plot the regression line and confidence interval on, by default None.
    delta : np.ndarray, optional
        An optional array indicating upper limits (1 for DETECTION, 0 for upper limit) to
        be passed to LinMix for proper handling of censored data, by default None.
    log_transformed : bool, optional
        Whether the data has been log-transformed, which affects how upper limits should be handled in
            LinMix, by default False.

    Returns
    -------
    tuple[float, float, float, float, float]
        The slope, slope uncertainty, intercept, intercept uncertainty, and p-value of the fit.
    """
    if mcmc:
        # If fitting with upper bounds in log-space, we need to translate all the y values up
        # by an arbitrary amount (10, corresponding to 10 dex) to ensure that upper limits
        # are properly handled in log-space without resulting in -inf values that can cause issues for LinMix
        shift_up = log_transformed and delta is not None and np.any(delta == 0)
        if shift_up:
            print("Data is log-transformed with upper limits, shifting y values up by 10 dex for LinMix...")
            y = y + 10  # Shift log-transformed values up by 10 dex to avoid -inf for upper limits
        
        lm = LinMix(x, y, xsig=x_err, ysig=y_err, delta=delta, K=2)

        try:
            lm.run_mcmc(silent=True)

            # Shift the intercept chain back down by 10 dex if log-transformed with upper limits to get the correct intercept values in log-space
            if shift_up:
                print("Shifting intercept chain back down by 10 dex for correct log-space values...")
                lm.chain['alpha'] -= 10
            
            slope = np.mean(lm.chain['beta'])
            sloperr = np.std(lm.chain['beta'])
            inter = np.mean(lm.chain['alpha'])
            intererr = np.std(lm.chain['alpha'])

            # Calculate p-value using the slope posterior distribution
            p_value = get_mcmc_p_value(lm.chain)
        except ValueError as e:
            print(f"LinMix MCMC failed to converge: {e}")
            return None, None, None, None, None
        # Plot the result by taking a sample of the posterior distribution of slopes and intercepts to show the confidence interval
        if ax_in is not None:
            x_fit = np.linspace(np.min(x), np.max(x), 10, endpoint=True)
            y_fit_samples = np.array([lm.chain[i]['alpha'] + lm.chain[i]['beta'] * x_fit for i in range(0, len(lm.chain), 25)])
            ax_in.plot(x_fit, y_fit_samples.T, color='red', alpha=0.01)

        # Quick check - create a new figure just for this (won't interfere with your main plot)
        fig_hist, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
        ax1.hist(lm.chain['beta'], bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='black')
        ax1.set_title('Slope posterior')
        ax1.set_xlabel(r'$\beta$')
        ax1.axvline(0, color='r', ls='--', label='Zero')
        ax1.legend()
        ax2.hist(lm.chain['alpha'], bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='black')
        ax2.set_title('Intercept posterior')
        ax2.set_xlabel(r'$\alpha$')
        
        return slope, inter, sloperr, intererr, p_value  # Return order: slope, intercept, slope_err, intercept_err, p_value

    else:
        # Perform ODR regression as a fallback if MCMC is disabled
        def linear_model(B, x):
            return B[0] * x + B[1]
        data = RealData(x, y, sx=x_err, sy=y_err)
        model = Model(linear_model)
        odr = ODR(data, model, beta0=[0., 1.])
        output = odr.run()
        slope, intercept = output.beta
        slope_err, intercept_err = output.sd_beta
        # Calculate p-value using the t-statistic for the slope
        t_stat = slope / slope_err if slope_err > 0 else 0
        p_value = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=len(x) - 2))  # Two-tailed test
        return slope, slope_err, intercept, intercept_err, p_value

In [ ]:
from tangelo import plotting as tgplt

log_quantities = ["LYA_EW", "FWHMR", "DISPR", "CONT", "EW", "FWHM", "DISP", "VEXP_ZELDA"]

def check_correlations(line_property: str, lya_properties: list[str], lines: list[str],
                       abs_lines: list[str], megatab: aptb.Table, min_points: int = 10,
                       significance_thresh: float = 0.01, mcmc: bool = True, 
                       fit_upper_limits: bool = False, upper_limit: float = 3.0,
                       logify: bool = False, plot_upper_limits: bool = False,
                       save_fig: bool = False) -> dict:
    """
    Check for correlations between a given line property (e.g. EW, FWHM) and a list of Lyman alpha properties.

    Parameters
    ----------
    line_property : str
        The line property to check (e.g. "EW", "FWHM").
    lya_properties : list[str]
        List of Lyman alpha properties to check against (e.g. ["LYA_EW", "DISPR", "CONT", "ASYMR", 
        "FWHMR", "BRRATIO", "BRSEP"]).
    lines : list[str]
        List of lines to check (e.g. ["SiII1260", "CII1334", "SiIV1394", "SiIV1403", "CIV1548", 
        "HeII1640", "OIII1660", "CIII1907"]).
    abs_lines : list[str]
        List of lines that should be treated as absorption (e.g. ["SiII1260", "CII1334", 
        "SiIV1394", "SiIV1403"]).
    megatab : aptb.Table
        The megatable containing the data.
    min_points : int, optional
        Minimum number of points required to attempt fitting a correlation, by default 10.
    significance_thresh : float, optional
        P-value (or Bayesian equivalent) threshold for determining significant correlations, by default 0.01.
    mcmc : bool, optional
        Whether to use MCMC regression for fitting the correlation, by default True.
    fit_upper_limits : bool, optional
        Whether to attempt to fit upper limits for non-detections, by default False.
    upper_limit : float, optional
        The sigma level to use for upper limits, by default 3.0.
    logify : bool, optional
        Whether to log-transform the line property for fitting, by default False.
    plot_upper_limits : bool, optional
        Whether to plot upper limits for non-detections, by default False.
    save_fig : bool, optional
        Whether to save the figure, by default False.


    Returns
    -------
    dict
        A dictionary containing the correlation summaries for each line and Lyman alpha property.
    """
    summaries = {}
    for line in lines:
        summaries[line] = {}
        for lya_prop in lya_properties:
            lya_col, lya_col_err = get_lya_property(megatab, lya_prop)

            # Prepare line EW column
            line_col, line_err = get_line_property(megatab, line, line_property, abs=line in abs_lines)

            # Prepare mask
            mask = prepare_scatter_mask(megatab, line, line_col, line_property, lya_col, lya_prop,
                                               abs_lines, include_upper_limits=fit_upper_limits)
            
            # x-axis is Lya property, y-axis is line property
            x_vals = lya_col[mask]
            y_vals = line_col[mask]
            x_errs = lya_col_err[mask]
            y_errs = line_err[mask]

            # Initialize detection mask: by default, all points are detections
            det_mask = np.ones_like(y_vals, dtype=bool)
            
            # Generate a boolean array for the upper limits and replace any such values with 
            # the specified sigma upper limit value
            nondet_mask = np.zeros_like(y_vals, dtype=bool)
            if fit_upper_limits:
                 # Identify non-detections based on the specified sigma threshold
                nondet_mask = y_vals < upper_limit * y_errs
                # Identify detections based on the specified sigma threshold
                det_mask = y_vals >= upper_limit * y_errs
                # Ensure that n_det + n_nondet = total number of points after masking
                assert np.sum(det_mask) + np.sum(nondet_mask) == len(y_vals), \
                        "Error in non-detection masking: number of detections + non-detections does not" \
                        " equal total number of points after masking."
                y_vals[~det_mask] = upper_limit * y_errs[~det_mask]  # Replace non-detections with specified sigma upper limit
 
            # Transform EWs and FWHMs/DISP into log space
            plot_log_x = False
            plot_log_y = False
            if lya_prop in log_quantities and logify:
                x_vals_orig = x_vals.copy()  # Keep a copy of the original values for error transformation
                x_vals = np.log10(x_vals)
                x_errs = x_errs / (x_vals_orig * np.log(10))
                plot_log_x = True
            if line_property in log_quantities and logify:
                y_vals_orig = y_vals.copy()  # Keep a copy of the original values for error transformation
                y_vals = np.log10(y_vals)
                y_errs = y_errs / (y_vals_orig * np.log(10))
                plot_log_y = True

            # Raise error if there are negative error bars in log space, which can happen if there are negative 
            # values or values consistent with zero.
            if np.any(x_errs < 0) or np.any(y_errs < 0):
                print(f"Warning: Negative error bars detected for {line} {line_property} vs {lya_prop}. Skipping"
                    " correlation analysis.")
                print(x_errs[x_errs < 0], y_errs[y_errs < 0])
                continue

            # Make scatter plot
            fig, ax = plt.subplots(figsize=(6, 4))

            make_scatter(
                [x_vals, x_errs],
                [y_vals, y_errs],
                ax=ax,
                z=megatab['z'][mask],
                upper_bounds = nondet_mask
            )

            # Only attempt to fit a line if there are enough valid points after masking to avoid unreliable
            #  fits and overinterpreting small number statistics
            if np.sum(mask) >= min_points:

                # Final check on data quality: look for NaNs, negative error bars, or infinite values
                for array in [x_vals, y_vals, x_errs, y_errs]:
                    # First we check all arrays for NaN and inf
                    if np.any(np.isnan(array)):
                        print(f"Warning: NaN values detected in data for {line} {line_property} vs {lya_prop}. Skipping"
                            " correlation analysis.")
                        print(array[np.isnan(array)])
                        plt.close(fig)  # Close the plot if data quality issues are detected to avoid clutter
                        continue
                    if np.any(np.isinf(array)):
                        print(f"Warning: Infinite values detected in data for {line} {line_property} vs {lya_prop}. Skipping"
                            " correlation analysis.")
                        print(array[np.isinf(array)])
                        plt.close(fig)  # Close the plot if data quality issues are detected to avoid clutter
                        continue
                for array in [x_errs, y_errs]:
                    # Then we check the error arrays for negative values, which can occur after log transformation if there are values consistent with zero or negative values in the original data
                    if np.any(array < 0):
                        print(f"Warning: Negative error bars detected for {line} {line_property} vs {lya_prop}. Skipping"
                            " correlation analysis.")
                        print(array[array < 0])
                        plt.close(fig)  # Close the plot if data quality issues are detected to avoid clutter
                        continue

                # Do a preliminary fit to pre-screen for significant correlations before attempting the more computationally expensive MCMC fit
                _, _, _, prelim_p_value, _ = linregress(x_vals[det_mask], y_vals[det_mask])
                if prelim_p_value >= significance_thresh:
                    print(f"No significant correlation found for {line} {line_property} vs {lya_prop} (prelim p={prelim_p_value:.3e}). Skipping MCMC fit.")
                    plt.close(fig)  # Close the plot if not significant to avoid clutter
                    continue
                elif prelim_p_value < significance_thresh and mcmc:
                    print(f"Preliminary fit suggests significant correlation for {line} {line_property} vs "
                        f"{lya_prop} (prelim p={prelim_p_value:.3e}). Proceeding with MCMC fit.")
                elif prelim_p_value < significance_thresh and not mcmc:
                    print(f"Preliminary fit suggests significant correlation for {line} {line_property} vs "
                        f"{lya_prop} (prelim p={prelim_p_value:.3e}). Proceeding with ODR fit.")

                # Fit line to original data with MCMC: line_property (y) vs lya_property (x)
                slope, intercept, slope_err, intercept_err, p_value = do_linregress(x_vals, y_vals, x_errs, y_errs, 
                                                                                    delta=det_mask, mcmc=mcmc,
                                                                                    ax_in=ax, log_transformed=plot_log_y)
            
                # Plot best-fit line
                print(f"Correlation for {line} {line_property} vs {lya_prop}: slope={slope:.2f}±{slope_err:.2f}, "
                    f"intercept={intercept:.2f}±{intercept_err:.2f}, p={p_value:.3e}")
                x_fit = np.linspace(np.min(x_vals), np.max(x_vals), 100)
                y_fit = slope * x_fit + intercept
                ax.plot(x_fit, y_fit, color='red', 
                        label=f"Slope={slope:.2f}±{slope_err:.2f}\n"
                        f"Intercept={intercept:.2f}±{intercept_err:.2f}\n"
                        f"(p={p_value:.3e})")
                ax.legend()
            else:
                print(f"Not enough points to fit line for {line} {line_property} vs {lya_prop} (n={np.sum(mask)})")
                plt.close(fig)  # Close the plot if not enough points to avoid clutter
                continue


            ax.set_xlabel(f"{'log ' if plot_log_x else ''}{tgplt.get_plot_name(lya_prop)}")
            ax.set_ylabel(f"{'log ' if plot_log_y else ''}{tgplt.get_plot_name(line)} {tgplt.get_plot_name(line_property)}")
            ax.set_title(f"{tgplt.get_plot_name(line, unit=False)} {tgplt.get_plot_name(line_property, unit=False)} "
                         f"vs {tgplt.get_plot_name(lya_prop, unit=False)}")

            # ax.set_xscale('log') if plot_log_x else ax.set_xscale('linear')
            # if np.size(line_col[mask]) > 0:  # Only set yscale if there are valid points to avoid warnings
            #     ax.set_yscale('log') if plot_log_y else ax.set_yscale('linear')

            if save_fig:
                fig.savefig(f"plots/{line}_{line_property}_vs_{lya_prop}.png", dpi=300, bbox_inches='tight')

            plt.show()

            summaries[line][lya_prop] = {
                'slope': slope,
                'slope_err': slope_err,
                'intercept': intercept,
                'intercept_err': intercept_err,
                'p_value': p_value,
                'n_points': np.sum(mask)
            }
    return summaries

In [ ]:
# Plot fitted line rest-frame EWs against Lyman alpha properties


lines_to_plot = [
    "SiII1260",
    "CII1334",
    "SiIV1394",
    "SiIV1403",
    "CIV1548",
    "HeII1640",
    "OIII1660",
    "CIII1907",
]

lya_properties = [
    "LYA_EW",
    "DISPR",
    "CONT",
    "ASYMR",
    "FWHMR",
    "BRRATIO",
    "BRSEP",
    "DELTAV_LYA",
    "VEXP_ZELDA",
    "LOGN_ZELDA",
]

# list to specify which lines should be treated as absorption
abs_lines = [
    "SiII1260",
    "CII1334",
    "SiIV1394",
    "SiIV1403",
]

line_properties = ["EW", "FWHM", "CVEL"]

summaries = {}
for line_prop in line_properties:
    fit_upper_limits = line_prop in ["EW"]
    summaries[line_prop] = check_correlations(line_prop, lya_properties, lines_to_plot, abs_lines, megatab,
                                                min_points=6, significance_thresh=0.05, mcmc=True, 
                                                fit_upper_limits=False, upper_limit=3.0, logify=True,
                                                plot_upper_limits=True, save_fig=True)


In [ ]:
# Print summary of significant correlations
for line_prop, line_summaries in summaries.items():
    print(f"\nSummary of correlations for {line_prop}:")
    for line, lya_summaries in line_summaries.items():
        for lya_prop, summary in lya_summaries.items():
            if summary['p_value'] < 0.05:
                print(f"{line} {line_prop} vs {lya_prop}: slope={summary['slope']:.5f}±{summary['slope_err']:.5f}, "
                    f"intercept={summary['intercept']:.5f}±{summary['intercept_err']:.5f}, "
                    f"p={summary['p_value']:.3e}, n={summary['n_points']}")

In [ ]:
from typing import Union

def ks_test_lya_mc(line: str, lya_prop: str, megatab: aptb.Table, line_property: str="EW", 
                   abs_lines: list = abs_lines, threshold: Union[float, str] ='auto',
                   significance_thresh: float = 3.0) -> tuple[float, float]:
    """
    Perform a KS test comparing the distribution of a given lyman alpha property
    (e.g. DISPR, FWHMR, ASYMR) for sources with high vs low values of a given line property (e.g. EW).
    Handles cases of upper limits in the line property by including in the low-value group
    all sources with detections OR upper limits below the required threshold.

    Parameters
    ----------
    line : str
        The line to analyze (e.g., "SiII1260").
    lya_prop : str
        The Lyman alpha property to split on (e.g., "LYA_EW").
    megatab : aptb.Table
        The megatable containing the data.
    line_property : str, optional
        The line property to use for splitting the sample (e.g. "EW", "FWHM"), by default "EW".
    abs_lines : list, optional
        List of lines that should be treated as absorption, by default abs_lines defined above.
    threshold : float or str, optional
        The threshold to use for splitting the sample into high vs low Lyman alpha property groups. If 'auto', will use the median value of the Lyman alpha property for splitting, by default 'auto'.
    significance_thresh : float, optional
        The sigma threshold to use for including sources based on their SNR, by default 3.0.

    Returns
    -------
    tuple[float, float]
        The KS statistic and p-value for the test.
    """
    # Get the line property column and the Lyman alpha property column
    line_col, line_err = get_line_property(megatab, line, line_property, abs=line in abs_lines)
    lya_col, lya_col_err = get_lya_property(megatab, lya_prop)

    # Prepare mask for valid data points
    mask = prepare_scatter_mask(megatab, line, line_col, line_property, lya_col, lya_prop,
                                               abs_lines, include_upper_limits=True)
    
    significance_mask = (megatab[f"SNR_{line}"].copy() > significance_thresh)  # Mask for significant detections
    if line in abs_lines:
        # For absorption lines, we want to flip the sign of the SNR threshold to identify significant absorptions
        significance_mask = (megatab[f"SNR_{line}"].copy() < -significance_thresh)
    
    detections = mask & significance_mask  # Significant detections
    upper_limits = mask & (~significance_mask)  # Non-detections or upper limits

    # Replace non-detections with 3 sigma upper bounds
    line_col[upper_limits] = 3 * line_err[upper_limits]

    # Split the data into two groups based on the median value of the Lyman alpha property or provided threshold
    if threshold == 'auto':
        median_line = np.nanmedian(line_col)
        print(f"Using median {line} {line_property} value of {median_line:.2f} for splitting groups.")
    else:
        median_line = threshold
    group_high = lya_col[detections & (line_col >= median_line)]
    group_low = lya_col[(detections & (line_col < median_line)) | (upper_limits & (line_col < median_line))]

    # Logify the quantities if they are in the log_quantities list to ensure that the KS test is comparing the same transformed distributions as the correlation analysis
    if lya_prop in log_quantities:
        group_high = np.log10(group_high)
        group_low = np.log10(group_low)

    # If either group is empty, we cannot perform the KS test
    if len(group_high) == 0 or len(group_low) == 0:
        print(f"Warning: One of the groups for {line} {line_property} vs {lya_prop} is empty. Cannot perform KS test.")
        return np.nan, np.nan

    # Perform KS test comparing the line property distributions of the two groups
    ks_statistic, p_value = stats.ks_2samp(group_high, group_low)

    if p_value < 0.05:
        print(f"Significant difference in {lya_prop} distribution for high vs low {line} {line_property} (KS p={p_value:.3e}).")
    else:
        print(f"No significant difference in {lya_prop} distribution for high vs low {line} {line_property} (KS p={p_value:.3e}).")
        return ks_statistic, p_value  # Skip plotting if not significant to avoid clutter

    print(f"KS test for {line} {line_property} vs {lya_prop}: KS statistic={ks_statistic:.3f}, p-value={p_value:.3e}")

    n_bins = 20
    bins = np.linspace(min(np.min(group_high), np.min(group_low)), max(np.max(group_high), np.max(group_low)), n_bins)

    # Plot comparative histograms of the Lyman alpha property for the two groups
    plt.figure(figsize=(6, 4))
    plt.hist(group_high, bins=bins, alpha=0.7, label=f"{line} {line_property} $\geq {median_line:.2f}$", 
             color='steelblue', edgecolor='black')
    plt.hist(group_low, bins=bins, alpha=0.7, label=f"{line} {line_property} $< {median_line:.2f}$", 
             color='salmon', edgecolor='black')
    plt.xlabel(tgplt.get_plot_name(lya_prop))
    plt.ylabel("Number of sources")
    plt.title(f"{line} {line_property} vs {lya_prop}\nKS p-value={p_value:.3e}")
    plt.legend()
    plt.savefig(f"plots/{line}_{line_property}_vs_{lya_prop}_KS_hist.png", dpi=300, bbox_inches='tight')
    plt.show()  

    return ks_statistic, p_value

In [ ]:
# Now we perform two-sample KS tests to compare the distributions of Lyman alpha properties for sources
# above and below emission line EW thresholds

megatab_highlyasnr = megatab[megatab['SNRR'] > 10]  # Subset of sources with highly significant Lyman alpha detections

ks_results = {}
for line in lines_to_plot:
    ks_results[line] = {}
    for lya_prop in lya_properties:
        ks_statistic, p_value = ks_test_lya_mc(line, lya_prop, megatab_highlyasnr, line_property="EW", abs_lines=abs_lines,
                                               threshold='auto')
        ks_results[line][lya_prop] = {
            'ks_statistic': ks_statistic,
            'p_value': p_value
        }

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

# =============================================================================
# Feature matrix construction
# =============================================================================
# "Complete" features (every source has these — Lya always detected)
lya_feature_defs = {
    'LYA_EW'   : lambda t: t['FLUXR'] / t['CONT'],
    'FWHMR'    : lambda t: t['FWHMR'],
    'DISPR'    : lambda t: t['DISPR'],
    'ASYMR'    : lambda t: t['ASYMR'],
    'BRRATIO'  : lambda t: t['FLUXB'] / t['FLUXR'],
    'CONT'     : lambda t: t['CONT'],
}

# Sparse line EWs (sign-flipped for absorption lines so all should be positive)
line_feature_defs = {
    'SiII1260' : dict(col='SiII1260',  is_abs=True),
    'CII1334'  : dict(col='CII1334',   is_abs=True),
    'SiIV1394' : dict(col='SiIV1394',  is_abs=True),
    'SiIV1403' : dict(col='SiIV1403',  is_abs=True),
    'CIV1548'  : dict(col='CIV1548',   is_abs=False),
    'HeII1640' : dict(col='HeII1640',  is_abs=False),
    'OIII1660' : dict(col='OIII1660',  is_abs=False),
    'CIII1907' : dict(col='CIII1907',  is_abs=False),
}

SNR_DET_THRESH  =  3.0   # |SNR| > this => detection
SNR_UL_THRESH   =  1.0   # |SNR| < this => treat as upper limit (not just noise)
N_COMPONENTS    =  3     # Number of latent factors to extract

# =============================================================================
# Build arrays
# =============================================================================
N = len(megatab)
n_lya  = len(lya_feature_defs)
n_line = len(line_feature_defs)
n_feat = n_lya + n_line

feature_names = list(lya_feature_defs.keys()) + list(line_feature_defs.keys())

X       = np.full((N, n_feat), np.nan)   # raw values (log-space where appropriate)
X_ul    = np.full((N, n_feat), np.nan)   # upper-limit values
is_obs  = np.zeros((N, n_feat), dtype=bool)   # True = detected
is_ul   = np.zeros((N, n_feat), dtype=bool)   # True = upper limit (not detected but constrained)

# --- Lya features (always observed) ---
for j, (name, func) in enumerate(lya_feature_defs.items()):
    vals = np.array(func(megatab), dtype=float)
    finite = np.isfinite(vals) & (vals > 0)
    X[finite, j]      = np.log10(vals[finite])
    is_obs[finite, j] = True

# --- Line EWs ---
for j, (name, defn) in enumerate(line_feature_defs.items(), start=n_lya):
    col    = defn['col']
    is_abs = defn['is_abs']

    flux     = np.array(megatab[f'FLUX_{col}'],     dtype=float)
    flux_err = np.array(megatab[f'FLUX_ERR_{col}'], dtype=float)
    cont     = np.array(megatab[f'CONT_{col}'],     dtype=float)
    cont_err = np.array(megatab[f'CONT_ERR_{col}'], dtype=float)
    snr      = np.array(megatab[f'SNR_{col}'],      dtype=float)
    try:
        flag = np.array(megatab[f'FLAG_{col}'])
        unflagged = (flag == '')
    except KeyError:
        unflagged = np.ones(N, dtype=bool)

    ew     = flux / cont
    if is_abs:
        ew *= -1   # make absorption EWs positive
        det_mask = snr < -SNR_DET_THRESH
        ul_mask  = (snr >= -SNR_DET_THRESH) & (snr <= SNR_UL_THRESH)
    else:
        det_mask = snr > SNR_DET_THRESH
        ul_mask  = (snr >= -SNR_UL_THRESH) & (snr <= SNR_DET_THRESH)

    cont_ok  = (cont / cont_err > 3.0) if np.any(cont_err > 0) else np.ones(N, dtype=bool)
    det_mask &= unflagged & cont_ok & (ew > 0)
    ul_mask  &= cont_ok

    # 3-sigma upper limit on |EW| from the flux error
    ew_ul = 3 * np.abs(flux_err) / np.abs(cont)

    X[det_mask, j]      = np.log10(ew[det_mask])
    is_obs[det_mask, j] = True

    # Upper-limit: set UL value; X stays NaN so imputer will fill it
    X_ul[ul_mask, j]  = np.log10(np.maximum(ew_ul[ul_mask], 1e-6))
    is_ul[ul_mask, j] = True

print("Data matrix built.")
print(f"  Sources: {N}")
print(f"  Features: {n_feat}  ({n_lya} Lya + {n_line} line EWs)")
for j, name in enumerate(feature_names):
    n_det = is_obs[:, j].sum()
    n_ul  = is_ul[:, j].sum()
    n_mis = N - n_det - n_ul
    print(f"  {name:<12}  det={n_det:>4}  UL={n_ul:>4}  missing={n_mis:>4}")

# =============================================================================
# Imputation: MICE anchored by the always-observed Lya features
# =============================================================================
# IterativeImputer uses chained regression to fill NaN.
# With Lya features always observed they anchor every imputation round.
imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=20,
    random_state=42,
    min_value={j: -3 for j in range(n_feat)},   # sensible log-EW floor
    max_value={j:  3 for j in range(n_feat)},   # sensible log-EW ceiling
)
X_imp = imputer.fit_transform(X)

# Clamp upper-limit entries: if imputed value > log(UL), replace with log(UL)
ul_idx = np.where(is_ul & ~is_obs)
X_imp[ul_idx] = np.minimum(X_imp[ul_idx], X_ul[ul_idx])

print("\nImputation complete.")

# =============================================================================
# Factor Analysis (EM-based, more robust than PCA for heteroscedastic data)
# =============================================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imp)

fa = FactorAnalysis(n_components=N_COMPONENTS, random_state=42, max_iter=2000)
scores = fa.fit_transform(X_scaled)       # (N, n_components)
loadings = fa.components_.T               # (n_feat, n_components)

# Explained variance (approximated as variance of projected scores)
total_var     = np.sum(np.var(X_scaled, axis=0))
explained_var = np.var(scores, axis=0)
explained_frac = explained_var / total_var
print(f"\nApproximate explained variance per factor:")
for k in range(N_COMPONENTS):
    print(f"  Factor {k+1}: {100*explained_frac[k]:.1f}%")

# =============================================================================
# Plots
# =============================================================================

# --- 1: Loadings heatmap ---
fig, ax = plt.subplots(figsize=(8, 5), facecolor='w')
im = ax.imshow(loadings, aspect='auto', cmap='RdBu_r',
               vmin=-np.abs(loadings).max(), vmax=np.abs(loadings).max())
ax.set_xticks(range(N_COMPONENTS))
ax.set_xticklabels([f'Factor {k+1}' for k in range(N_COMPONENTS)])
ax.set_yticks(range(n_feat))
ax.set_yticklabels(feature_names)
ax.set_title('Factor loadings')
for i in range(n_feat):
    ax.axhline(i + 0.5, color='gray', linewidth=0.4, alpha=0.5)
ax.axhline(n_lya - 0.5, color='k', linewidth=1.5)   # separator between Lya and line EWs
plt.colorbar(im, ax=ax, label='Loading')
plt.tight_layout()
plt.show()
plt.close(fig)

# --- 2: Biplot: scores + loading arrows for factors 1 and 2 ---
lya_ew_col = np.array(megatab['FLUXR'] / megatab['CONT'], dtype=float)
lya_ew_log = np.where((lya_ew_col > 0) & np.isfinite(lya_ew_col),
                      np.log10(lya_ew_col), np.nan)

fig, ax = plt.subplots(figsize=(8, 7), facecolor='w')
sc = ax.scatter(scores[:, 0], scores[:, 1],
                c=lya_ew_log, cmap='plasma', alpha=0.6, s=20, zorder=2)
plt.colorbar(sc, ax=ax, label=r'$\log_{10}$(Ly$\alpha$ EW)')

# Arrow scale: normalise so longest arrow fits inside the score cloud
scale = 0.4 * max(np.ptp(scores[:, 0]), np.ptp(scores[:, 1]))
arrow_len = np.sqrt(loadings[:, 0]**2 + loadings[:, 1]**2)
max_len   = arrow_len.max()
for i, name in enumerate(feature_names):
    dx = loadings[i, 0] / max_len * scale
    dy = loadings[i, 1] / max_len * scale
    color = 'royalblue' if i < n_lya else 'crimson'
    ax.annotate('', xy=(dx, dy), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
    ax.text(dx * 1.12, dy * 1.12, name, fontsize=8, color=color, ha='center')

ax.axhline(0, color='k', linewidth=0.5, alpha=0.4)
ax.axvline(0, color='k', linewidth=0.5, alpha=0.4)
ax.set_xlabel(f'Factor 1 ({100*explained_frac[0]:.1f}% var.)')
ax.set_ylabel(f'Factor 2 ({100*explained_frac[1]:.1f}% var.)')
ax.set_title('Biplot  [blue = Ly$\\alpha$ features, red = line EWs]')
plt.tight_layout()
plt.show()
plt.close(fig)

# --- 3: Factor scores vs individual Lya properties ---
lya_plot_cols = {
    'LYA_EW' : lya_ew_log,
    'FWHMR'  : np.log10(np.array(megatab['FWHMR'], dtype=float)),
    'ASYMR'  : np.array(megatab['ASYMR'], dtype=float),
    'BRRATIO': np.log10(np.clip(np.array(megatab['FLUXB'] / megatab['FLUXR'], dtype=float), 1e-4, None)),
}

from scipy.stats import spearmanr
n_lya_plot = len(lya_plot_cols)
fig, axes = plt.subplots(N_COMPONENTS, n_lya_plot,
                          figsize=(4 * n_lya_plot, 3.5 * N_COMPONENTS),
                          facecolor='w')
axes = np.atleast_2d(axes)

for k in range(N_COMPONENTS):
    for j, (prop_name, prop_vals) in enumerate(lya_plot_cols.items()):
        ax = axes[k, j]
        good = np.isfinite(prop_vals) & np.isfinite(scores[:, k])
        ax.scatter(prop_vals[good], scores[good, k], alpha=0.4, s=15, color='steelblue')
        if good.sum() > 4:
            rho, p = spearmanr(prop_vals[good], scores[good, k])
            ax.text(0.04, 0.95, f'ρ = {rho:.2f}\np = {p:.2g}',
                    transform=ax.transAxes, va='top', fontsize=9,
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
        ax.set_xlabel(prop_name, fontsize=9)
        if j == 0:
            ax.set_ylabel(f'Factor {k+1} score', fontsize=9)

plt.suptitle('Factor scores vs Lyman-alpha properties', y=1.01)
plt.tight_layout()
plt.show()
plt.close(fig)
